In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

sys.path.insert(0, str(find_project_root() / "src"))
import importlib
import modeling as m
importlib.reload(m)

dev = m.load("2017-2020")

MODELS = {
    "Logistic": m.make_logistic,
    "Random forest": m.make_random_forest,
    "XGBoost": m.make_xgboost,
}
SETS = {"A": m.MODEL_A, "B": m.MODEL_B}
COLUMNS = ["roc_auc", "pr_auc", "brier", "cal_slope", "obs_exp",
           "specificity", "referred_pct", "screen_per_case"]

rows = {}
for set_name, predictors in SETS.items():
    for model_name, make in MODELS.items():
        start = time.time()
        print(f"Model {set_name} - {model_name} ...", flush=True)
        result = m.cross_validate(make, predictors, dev, verbose=True)
        s = m.summarise(result)
        rows[(set_name, model_name)] = s.loc[COLUMNS, "mean"]
        rows[(set_name, model_name)]["auc_low"] = s.loc["roc_auc", "low"]
        rows[(set_name, model_name)]["auc_high"] = s.loc["roc_auc", "high"]
        print(f"  finished in {(time.time() - start) / 60:.1f} min\n", flush=True)

table = pd.DataFrame(rows).T
table.index.names = ["set", "model"]
print(table.round(3))

Model A - Logistic ...
  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
  finished in 0.2 min

Model A - Random forest ...
  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
  finished in 10.4 min

Model A - XGBoost ...
  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
  finished in 5.3 min

Model B - Logistic ...
  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
  finished in 0.3 min

Model B - Random forest ...
  repeat 1/10 done
